# Spine XR Augmentation — Project 3 (Colab pipeline)

Replicating the methodology of *"Analysis of Augmentation Techniques for Spine X-Ray Images"* (Elakiya & Anand, 2026) on the VinDr-SpineXR dataset, with two engineering corrections (held-out test set; Macro F1 instead of accuracy).

**This notebook covers Milestones 1 + 2:**
1. Phase 01 — audit (build per-image multi-label tables).
2. Phase 02 — split into Cases 1–4 (paper-faithful + Case 4 stress-test).
3. Phase 03 — baseline (no-augmentation) classifier per Case × {VGG16, InceptionV3}.

Designed for **Colab Pro+ with A100 GPU**. All artefacts (splits, weights, metrics) are written under `OUTPUTS_ROOT` so that they survive across Colab sessions if you point it at Drive.

## 0. Environment setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === USER-CONFIGURABLE PATHS ===
# Adjust to match where you have uploaded the repo and the dataset on Drive.

# 1) PROJECT_ROOT must contain configs/, scripts/, src/ (this notebook should be inside notebooks/).
PROJECT_ROOT = '/content/drive/MyDrive/spine-xr-augmentation-study-3'

# 2) DATASET_ROOT must contain abnormal/{train_pngs,test_pngs,*.csv} and normal/{train_pngs,test_pngs,*.csv}
#    i.e. the same VinDr-SpineXR layout used in study-2.
DATASET_ROOT = '/content/drive/MyDrive/vindr-spinexr-png'

# 3) Where to write artefacts. Drive = persistent (slow per-write, big checkpoints take time).
#    Local /content = fast but lost on session disconnect.
#    Default: Drive for splits + small files; weights synced explicitly at the end.
OUTPUTS_ROOT = f'{PROJECT_ROOT}/outputs'

# 4) For training-time speed, keep heavy outputs (best.pth) on local disk and sync to Drive at the end.
FAST_OUTPUTS_LOCAL = '/content/outputs_local'
USE_LOCAL_OUTPUTS_FOR_TRAINING = True   # set False if you want all outputs directly on Drive

import os
assert os.path.isdir(PROJECT_ROOT), f'PROJECT_ROOT not found: {PROJECT_ROOT}'
assert os.path.isdir(DATASET_ROOT), f'DATASET_ROOT not found: {DATASET_ROOT}'
os.makedirs(OUTPUTS_ROOT, exist_ok=True)
if USE_LOCAL_OUTPUTS_FOR_TRAINING:
    os.makedirs(FAST_OUTPUTS_LOCAL, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('OUTPUTS_ROOT:', OUTPUTS_ROOT)

In [ ]:
%cd $PROJECT_ROOT
!ls

In [ ]:
# Install Python deps (Colab already has torch + torchvision + opencv + numpy + pandas + sklearn).
# Albumentations 2.x is newer than what Colab ships by default; pin to 2.0.x for the new Pad API.
!pip install -q 'albumentations>=2.0,<3.0' tabulate pyyaml

In [ ]:
# GPU sanity check
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 1. Build a Colab-specific config

We don't edit `configs/base.yaml` (which holds the default local layout). Instead we synthesise `configs/base_colab.yaml` here, with paths pointing at the real Drive locations. Every Phase script accepts `--config configs/base_colab.yaml`.

In [ ]:
import yaml
from pathlib import Path

training_outputs = FAST_OUTPUTS_LOCAL if USE_LOCAL_OUTPUTS_FOR_TRAINING else OUTPUTS_ROOT

colab_cfg = {
    'project': {'name': 'spine-xr-augmentation-study-3', 'seed': 42, 'num_workers': 4},
    'paths': {
        'dataset_root': DATASET_ROOT,
        'abnormal_train_pngs':       f'{DATASET_ROOT}/abnormal/train_pngs',
        'normal_train_pngs':         f'{DATASET_ROOT}/normal/train_pngs',
        'abnormal_test_pngs':        f'{DATASET_ROOT}/abnormal/test_pngs',
        'normal_test_pngs':          f'{DATASET_ROOT}/normal/test_pngs',
        'abnormal_train_annotations':f'{DATASET_ROOT}/abnormal/train_annotations.csv',
        'abnormal_test_annotations': f'{DATASET_ROOT}/abnormal/test_annotations.csv',
        'normal_train_metadata':     f'{DATASET_ROOT}/normal/train_metadata.csv',
        'normal_test_metadata':      f'{DATASET_ROOT}/normal/test_metadata.csv',
        'outputs_root': training_outputs,
    },
    'classes': [
        'Osteophytes',
        'Disc space narrowing',
        'Other lesions',
        'Foraminal stenosis',
        'Surgical implant',
        'Spondylolysthesis',
        'Vertebral collapse',
        'No finding',
    ],
}

Path('configs/base_colab.yaml').write_text(yaml.safe_dump(colab_cfg, sort_keys=False))
print('Wrote configs/base_colab.yaml. outputs_root =', training_outputs)
!cat configs/base_colab.yaml

## 2. Phase 01 — Audit (build per-image multi-label tables)

Output: `outputs/01_audit/{train,test}_labels.csv`, `audit_report.md`, plus per-class counts and co-occurrence matrices. The audit report is the canonical view of what's in the official VinDr split.

In [ ]:
!python scripts/01_audit.py --config configs/base_colab.yaml

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{training_outputs}/01_audit/audit_report.md').read())

## 3. Phase 02 — Per-Case splits

Each Case gets `train.csv` / `internal_val.csv` / `test.csv` plus `manifest.json`. Decisions baked in:
- D1: keep all positive-bearing images; cap NF at 1000 (study-stratified subsample).
- D2: multi-label inside a case (an image with both DSN and VC counts for both).
- D3: VinDr official train/test split is used as-is.
- D8: 10% study-disjoint internal-val carve-out from train (guardrail; not the test set).

Assertions enforced: zero study leakage between train ↔ internal_val ↔ test; ≥1 positive per case-class in test; every path exists on disk.

In [ ]:
!python scripts/02_data_splitter.py --config configs/base_colab.yaml --cases configs/cases.yaml

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{training_outputs}/02_splits/splits_summary.md').read())

## 4. Phase 03 — Baseline classifier (no augmentation)

**Per Case × {VGG16, InceptionV3}**: train multi-label sigmoid + BCE for 25 epochs, save `best.pth` on the highest **Macro F1 over the official test set**. The internal_val Macro F1 is logged each epoch as a guardrail.

Outputs per cell: `outputs/03_baseline/<case>/<backbone>/{best.pth, log.csv, metrics.json, test_metrics.md}`.

**Time budget on A100 (rough estimate):** ~6–10 min/case for VGG16, ~8–12 min/case for InceptionV3 at 25 epochs. Full sweep (4 cases × 2 backbones) ≈ 60–90 min. You can break it up by passing `--cases-filter` and `--backbones-filter`.

In [ ]:
# Smoke test first: 1 epoch, 1 case, 1 backbone — just to confirm dataloader + GPU + saving work end-to-end.
# Skip this cell once smoke has passed once.
!python scripts/03_train_classifier.py \
    --config configs/base_colab.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter vgg16 \
    --epochs 1 \
    --out-tag 03_smoke

In [ ]:
# Full Phase 03 sweep — all 4 cases, both backbones, 25 epochs.
!python scripts/03_train_classifier.py \
    --config configs/base_colab.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --out-tag 03_baseline

In [ ]:
# Aggregate summary across all (case, backbone) cells
from IPython.display import Markdown
Markdown(open(f'{training_outputs}/03_baseline/summary.md').read())

In [ ]:
# (Optional) per-cell test metrics, e.g. Case 1 / VGG16
from IPython.display import Markdown
Markdown(open(f'{training_outputs}/03_baseline/case_1/vgg16/test_metrics.md').read())

## 5. Persist results to Drive

If you trained into local `/content/outputs_local` for speed (default), copy the artefacts back to Drive so they survive session disconnects.

Note: copying `best.pth` files (~500 MB each for VGG16) can take a few minutes per file.

In [ ]:
if USE_LOCAL_OUTPUTS_FOR_TRAINING:
    import shutil, os
    src = FAST_OUTPUTS_LOCAL
    dst = OUTPUTS_ROOT
    print(f'syncing {src} -> {dst}')
    !mkdir -p "$dst"
    !rsync -ah --info=progress2 "$src/" "$dst/"
    print('done')
else:
    print('USE_LOCAL_OUTPUTS_FOR_TRAINING is False — outputs are already on Drive.')

## Next milestones (not yet in this notebook)

1. **Phase 04** — traditional augmentation (offline geometric expansion of real abnormals using Rotation 270° + Shear + per-case second rotation).
2. **Phase 05–07** — WGAN training (per minority class), snapshot+FID selection, sample generation.
3. **Phase 08** — hybrid set construction + classifier.
4. **Phase 09** — final reporting in the format of paper Tables 8/9/10, with Macro F1 alongside accuracy.

Each will get its own section appended to this notebook in subsequent milestones.